# **Day 67: Writing Our Own Modules and Packages**

Till now, all our code lived in notebooks. Real projects split the code into many `.py` files.

- **Module:** One `.py` file with functions and classes.
- **Package:** A **folder** of modules, with a special file named `__init__.py` inside.

**Real-life example:** A **module** is like one chapter of a book. A **package** is the whole book with many chapters.

## **Creating a Module**
The `%%writefile` command saves the cell's content into a file.

In [1]:
%%writefile rsutils.py
"""Small helper functions for Remote Sensing work."""

def ndvi(nir, red):
    """Normalized Difference Vegetation Index."""
    return (nir - red) / (nir + red) if (nir + red) else 0.0

def ndwi(green, nir):
    """Normalized Difference Water Index (water is positive)."""
    return (green - nir) / (green + nir) if (green + nir) else 0.0

def pixels_to_hectare(count, resolution_m=10):
    return count * resolution_m ** 2 / 10_000

print("rsutils loaded, __name__ =", __name__)

if __name__ == "__main__":
    print("Running rsutils directly: quick self test")
    assert round(ndvi(0.45, 0.05), 2) == 0.8
    assert ndwi(0.09, 0.02) > 0
    print("All tests passed")

Writing rsutils.py


In [2]:
import rsutils

print(round(rsutils.ndvi(0.45, 0.05), 3))
print(round(rsutils.ndwi(0.09, 0.02), 3))
print(rsutils.pixels_to_hectare(12500), "hectare")

rsutils loaded, __name__ = rsutils
0.8
0.636
125.0 hectare


## **What is `if __name__ == "__main__":`?**
Every Python file has a special variable called `__name__`:
- When the file is **imported** by another file, `__name__` is the **module's name** (`"rsutils"`).
- When the file is **run directly** (`python rsutils.py`), `__name__` is `"__main__"`.

So the code under `if __name__ == "__main__":` runs **only** when we run the file directly. It is a good place for quick tests and examples.

In [3]:
import subprocess, sys
result = subprocess.run([sys.executable, "rsutils.py"], capture_output=True, text=True)
print(result.stdout)

rsutils loaded, __name__ = __main__
Running rsutils directly: quick self test
All tests passed



When we imported it above, only "rsutils loaded" was printed. When we ran it directly, the self test also ran.

## **Creating a Package**
Let us make a small GIS package called `geokit` with this structure:
```
geokit/
    __init__.py       <- makes the folder a package
    distance.py       <- distance functions
    convert.py        <- coordinate conversion functions
```

In [4]:
import os
os.makedirs("geokit", exist_ok=True)

with open("geokit/__init__.py", "w") as f:
    f.write('"""geokit: a tiny GIS helper package."""\n'
            'from .distance import haversine\n'
            'from .convert import dms_to_dd\n'
            '__version__ = "0.1.0"\n')

with open("geokit/distance.py", "w") as f:
    f.write('''import math

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return round(2 * 6371 * math.asin(math.sqrt(a)), 2)
''')

with open("geokit/convert.py", "w") as f:
    f.write('''def dms_to_dd(d, m, s, hemisphere="N"):
    """Degrees, minutes, seconds to decimal degrees."""
    dd = d + m / 60 + s / 3600
    return round(-dd if hemisphere in "SW" else dd, 6)
''')

print(sorted(os.listdir("geokit")))

['__init__.py', 'convert.py', 'distance.py']


In [5]:
import geokit
from geokit import haversine
from geokit.convert import dms_to_dd

print("Version:", geokit.__version__)
print("Kolkata to Durgapur:", haversine(22.5726, 88.3639, 23.5204, 87.3119), "km")
print("Darjeeling latitude:", dms_to_dd(27, 2, 28))

Version: 0.1.0
Kolkata to Durgapur: 150.64 km
Darjeeling latitude: 27.041111


Let us understand the code above:
- `import geokit` runs `geokit/__init__.py`.
- Inside `__init__.py`, `from .distance import haversine` brings the function up to the package level. That is why `from geokit import haversine` works.
- The dot in `.distance` means "from the same package". This is called a **relative import**.

## **Reloading a Changed Module**
In a notebook, if we edit a `.py` file, Python does not see the change until we **reload** it.

In [6]:
import importlib

with open("rsutils.py", "a") as f:
    f.write("\ndef ndbi(swir, nir):\n    return (swir - nir) / (swir + nir)\n")

importlib.reload(rsutils)
print("NDBI (built-up index):", round(rsutils.ndbi(0.30, 0.20), 3))

rsutils loaded, __name__ = rsutils
NDBI (built-up index): 0.2


In [7]:
# See what is inside a module
print([name for name in dir(rsutils) if not name.startswith("__")])

['ndbi', 'ndvi', 'ndwi', 'pixels_to_hectare']


In [8]:
# Quick question
# If you run "python rsutils.py", what will __name__ be inside rsutils.py?